In [72]:
import torch
import torch.nn as nn
inputs = torch.tensor(
 [[0.43, 0.15, 0.89], # Your (x^1)
 [0.55, 0.87, 0.66], # journey (x^2)
 [0.57, 0.85, 0.64], # starts (x^3)
 [0.22, 0.58, 0.33], # with (x^4)
 [0.77, 0.25, 0.10], # one (x^5)
 [0.05, 0.80, 0.55]] # step (x^6)
)

class CausalAttention(nn.Module):
    def __init__(self, d_in, d_out,context_length,dropout, qkv_bias = False):
        super().__init__()
        # initialise the weights but with nn.linear
        self.d_out = d_out
        self.wq = nn.Linear(d_in,d_out, bias=qkv_bias)
        self.wk = nn.Linear(d_in,d_out, bias=qkv_bias)
        self.wv = nn.Linear(d_in,d_out, bias=qkv_bias)

        #dropouts and context length
        self.dropout = nn.Dropout(dropout)

        # masks too.
        self.register_buffer('mask', torch.triu(torch.ones(context_length, context_length),diagonal=1))

    def forward(self, inputs):
        batches, num_tokens, d_in = inputs.shape

        #keys, queries, values
        keys = self.wk(inputs)
        queries = self.wq(inputs)
        values = self.wv(inputs)

        # establishing relationships 
        attn_scores = queries @keys.transpose(1,2)

        # normalise scores and mask weights
        attn_scores.masked_fill_(self.mask.bool()[:num_tokens, :num_tokens], -torch.inf) # context length

        #attention weights
        attn_weights = torch.softmax(attn_scores/keys.shape[-1]**0.5, dim=-1)

        #dropout
        attn_weights = self.dropout(attn_weights)

        # context vectors
        context_vectors =  attn_weights @values

        return context_vectors
    

# Extending single-head attention to multi-head attention
- Our final step will be to extend the previously implemented causal attention class over multiple heads. This is also called multi-head attention.

- multi head refers to the mutliple heads with attention operating independently.

- a single causal attention module can be considered single-head attention, where there is only one set of attention
weights processing the input sequentially.

- We will build the multi head attention by stacking the casual attention modules.

# Stacking multiple single-head attention layers
![17. Multihead](../images/17.multihead-attention.png)

- In multi head attention, it's mainly the each layer/attention has its own weights and generate their own context vectors and then combining this all those context vectors.

- Using `multiple instances of the self-attention mechanism can be computationally intensive, but it’s crucial for the kind of complex pattern` recognition that models like transformer-based LLMs are known for.

- the main idea behind multi-head attention is to run the attention mechanism multiple times (in parallel) with different. `just like casual attention running paralelly for batches`.

# wrapper class for multihead attention

In [32]:
import torch.nn as nn
# class MultiHeadAttentionWrapper(nn.Module):
#     def __init__(self, d_in, d_out, context_length,dropout,num_heads ,qkv_bias=False):
#         self().__init__()
#         self.heads = nn.ModuleList([casualAttention(d_in, d_out, dropout,context_length,qkv_bias ) for _ in range(num_heads)])

#     def forward(self, x):
#         return torch.cat([head(x) for head in self.heads], dim=-1)

class MultiHeadAttentionWrapper(nn.Module):
    def __init__(self, d_in, d_out, context_length,dropout, num_heads, qkv_bias=False):
        super().__init__()
        self.heads = nn.ModuleList([CausalAttention(d_in, d_out, context_length, dropout, qkv_bias) for _ in range(num_heads)])
    def forward(self, x):
        return torch.cat([head(x) for head in self.heads], dim=-1)

        

- if we use this MultiHeadAttentionWrapper class with two attention heads
(via num_heads=2) and CausalAttention output dimension d_out=2, we get a fourdimensional context vector (d_out*num_heads=4)

![18. Multi head context vectors](../images/18.multihead-contextvec.png)

In [74]:
torch.manual_seed(123)
batch = torch.stack((inputs, inputs), dim=0)
context_length = batch.shape[1] # This is the number of tokens
d_in, d_out = 3, 2
mha = MultiHeadAttentionWrapper(d_in, d_out, context_length,0.0, num_heads=2)
context_vec = mha(batch)
context_vec
# final cvs are in the 4 cols due to d_out * no of heads.

NameError: name 'wq' is not defined

# Implementing multi-head attention with weight splits

- till now we have done the multihead attention wrapper done by wrapping the multiple single heads of the casual attention. this two classes can be done in single.
- Here we are merging the multiheadeattention wrapper with the casual attention.
- MultiHeadAttentionWrapper we created multiple heads by list of the casual attentions.

In [87]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout,num_heads, qkv_bias=False):
        super().__init__()
        assert (d_out % num_heads ==0), "d_out must be divisible by num_heads"
        self.d_out = d_out
        self.num_heads = num_heads
        self.dropout = nn.Dropout(dropout)
        # need a mask
        self.head_dim = d_out // num_heads
        self.out_proj = nn.Linear(d_out, d_out)
        #initialise weights
        self.wq = nn.Linear(d_in, d_out, bias = qkv_bias)
        self.wk = nn.Linear(d_in, d_out, bias = qkv_bias)
        self.wv = nn.Linear(d_in, d_out, bias = qkv_bias)

        #mask
        self.register_buffer("mask", torch.triu(torch.ones(context_length, context_length), diagonal = 1))

    def forward(self, inputs):
        # inputs now given in batches
        b, num_tokens, d_in = inputs.shape # gets a 3D array
        keys = self.wk(inputs)
        values = self.wq(inputs)
        queries = self.wv(inputs)

        # reshaping the matrix from (b, num_tokens, d_out) to (b, num_tokens, head_num, head_dim)
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)

        # Transposes from shape (b, num_tokens,num_heads, head_dim) to (b, num_heads, num_tokens, head_dim)
        keys = keys.transpose(1, 2)
        values = values.transpose(1, 2)
        queries = queries.transpose(1, 2)

        # now find relationships between qkv as previous.
        # Computes dot product for each head
        attention_scores = queries @ keys.transpose(2, 3)
        # mask
        mask_bool = self.mask.bool() [:num_tokens, : num_tokens]

        # apply -ve infinites fill in the upper triangle and add those.
        attention_scores.masked_fill_(mask_bool, -torch.inf)  # mask layer, filling num

        # needed softmax and normalisation
        attention_weights = torch.softmax(attention_scores/keys.shape[-1] ** 0.5, dim=-1)

        # add dropouts
        attention_weights = self.dropout(attention_weights)

        # Tensor shape: (b, num_tokens,n_heads,head_dim)
        context_vec = (attention_weights @ values).transpose(1, 2)

        # adds an optional linear projectiom
        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)

        # context vec
        context_vec = self.out_proj(context_vec)

        return context_vec
        

![19. Multihead](../images/19.multiheadNewVsOld.png)

- previously we used wrapper and individual heads processes independently. but now whole whole batch as big matrix processes and then splitted the outputs based on the num_heads in the final out_dim.

In [54]:
a = torch.tensor(
 [[
 [
 [0.2745, 0.6584, 0.2775, 0.8573],
 [0.8993, 0.0390, 0.9268, 0.7388],
 [0.7179, 0.7058, 0.9156, 0.4340]],
   
 [[0.0772, 0.3565, 0.1479, 0.5331],
 [0.4066, 0.2318, 0.4545, 0.9737],
 [0.4606, 0.5159, 0.4220, 0.5786]
 ]
 ]])
# 1 batch, 2 heads, 3 tokens, 4 d_out or head_dim

#we perform a batched matrix multiplication between the tensor itself and a view of the tensor where we transposed the last two dimensions, num_tokens and head_dim
print(a@ a.transpose(2, 3))

tensor([[[[1.3208, 1.1631, 1.2879],
          [1.1631, 2.2150, 1.8424],
          [1.2879, 1.8424, 2.0402]],

         [[0.4391, 0.7003, 0.5903],
          [0.7003, 1.3737, 1.0620],
          [0.5903, 1.0620, 0.9912]]]])


- initally the above transpositions made for the process of the actually matrix multiplications between the tokens, but the size of the array dimensions are updating so, to fetch those values properly, transpositions needed.
- later , the multiplications again the getting back to the same inital state, so again reshaped.

In [95]:
torch.manual_seed(123)
batch_size, context_length, d_in = batch.shape
d_out =3
mha = MultiHeadAttention(d_in, d_out, context_length, 0.0, num_heads= 3)

context_vecs = mha(batch)
context_vecs, batch.shape

(tensor([[[ 0.2767, -0.2800,  0.1826],
          [ 0.3007, -0.1976,  0.3338],
          [ 0.3096, -0.1733,  0.3835],
          [ 0.3047, -0.1634,  0.3810],
          [ 0.3183, -0.1785,  0.4037],
          [ 0.3073, -0.1610,  0.3913]],
 
         [[ 0.2767, -0.2800,  0.1826],
          [ 0.3007, -0.1976,  0.3338],
          [ 0.3096, -0.1733,  0.3835],
          [ 0.3047, -0.1634,  0.3810],
          [ 0.3183, -0.1785,  0.4037],
          [ 0.3073, -0.1610,  0.3913]]], grad_fn=<UnsafeViewBackward0>),
 torch.Size([2, 6, 3]))

# Exercise 3.3 Initializing GPT-2 size attention modules
- `Using the MultiHeadAttention class, initialize a multi-head attention module that has the same number of attention heads as the smallest GPT-2 model (12 attention heads). Also ensure that you use the respective input and output embedding sizes similar to GPT-2 (768 dimensions). Note that the smallest GPT-2 model supports a context length of 1,024 tokens.`

In [100]:
num_heads =12
d_in = 768
d_out = 768
num_tokens = 1024
context_length= nums_tokens

# first we need to create an array like this of size
# 1 batch 12 num_heads 1024 num tokens 768 dimensions in it . [1, 12, 1024, 768]

torch.manual_seed(123)
gpt_input = torch.rand(num_heads,num_tokens,d_out)
gpt_input.shape
mha = MultiHeadAttention(d_in, d_out, context_length, 0.0, num_heads)
context_vec = mha(gpt_input)
context_vec, context_vec.shape

(tensor([[[ 1.0900e-01,  3.5992e-01,  1.1084e-01,  ..., -1.9228e-01,
           -1.8152e-01,  1.2079e-01],
          [ 1.5918e-01,  3.3916e-01,  5.9100e-02,  ..., -1.3713e-01,
           -1.2775e-01,  1.5001e-03],
          [ 1.4513e-01,  3.5016e-01,  8.2591e-02,  ..., -1.1747e-01,
           -1.7284e-01,  1.0804e-02],
          ...,
          [ 1.8761e-01,  2.4431e-01,  6.9696e-02,  ..., -1.5409e-01,
           -1.6197e-01,  5.0131e-05],
          [ 1.8760e-01,  2.4422e-01,  6.9742e-02,  ..., -1.5425e-01,
           -1.6220e-01,  1.5256e-04],
          [ 1.8793e-01,  2.4405e-01,  6.9679e-02,  ..., -1.5459e-01,
           -1.6225e-01,  2.4115e-05]],
 
         [[ 3.7416e-01,  2.1388e-01,  2.2992e-01,  ..., -2.0745e-01,
           -2.5669e-01,  1.5607e-02],
          [ 2.0692e-01,  2.4164e-01,  1.3761e-01,  ..., -1.5852e-01,
           -2.7357e-01,  1.0567e-01],
          [ 2.0960e-01,  2.2597e-01,  1.5125e-01,  ..., -1.3435e-01,
           -2.3732e-01,  1.2489e-01],
          ...,
    